In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings 
warnings.simplefilter("ignore")

### How handle Missing
- Encoding episode_length as 40 to make it more represnetative of actual distribution since kind of a skew exists
- Try with some log transformations

# EDA

In [2]:
df = pd.read_csv("/kaggle/input/playground-series-s5e4/train.csv")
df.head()

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031


# Feature Engineering

### Transformations to Apply
#### Variable Specific Transformations
- Turning episode title into discrete numeric variable for the episode number
  
#### Numeric  
- Standardization
- Box-Cox Transformation for every numeric variable for funsies
  
#### Categorical  
- One hot encodings
  
#### Numeric and Categorical  
- Interaction Terms Between Every single variable, just for funsies

In [3]:
# # data cleaning
# def replace_episode_length(df):
#     df_temp = df.copy()

#     df_temp["Episode_Length_missing"] = pd.isna(df_temp["Episode_Length_minutes"])
#     df_temp.loc[df_temp["Episode_Length_minutes"].isna(), "Episode_Length_minutes"] = df_temp["Episode_Length_minutes"].median()
#     return df_temp

# def replace_guest_popularity(df):
#     df_temp = df.copy()

#     df_temp["Guest_Popularity_percentage"] = pd.isna(df_temp["Guest_Popularity_percentage"])
#     df_temp.loc[df_temp["Guest_Popularity_percentage"].isna(), "Guest_Popularity_percentage"] = df_temp["Guest_Popularity_percentage"].median()
#     return df_temp

# def replace_num_ads(df):
#     df_temp = df.copy()

#     df_temp["Number_of_Ads"] = pd.isna(df_temp["Number_of_Ads"])
#     df_temp.loc[df_temp["Number_of_Ads"].isna(), "Number_of_Ads"] = 0
#     return df_temp

# def replace_outliers(df):
#     df_temp = df.copy()

#     df_temp.loc[df_temp["Episode_Length_minutes"] > 300, "Episode_Length_minutes"] = 120
#     df_temp.loc[df_temp["Number_of_Ads"] > 10, "Number_of_Ads"] = 10
#     df_temp.loc[df_temp["Host_Popularity_percentage"] < 20] = 20
#     df_temp.loc[df_temp["Host_Popularity_percentage"] > 100] = 100
#     df_temp.loc[df_temp["Guest_Popularity_percentage"] > 100] = 100
    
#     return df_temp

# def clean_data(df):
#     df_temp = df.copy()
#     df_temp = df_temp.drop(columns=["id"])

#     # replacing outliers with max/min
#     df_temp = replace_outliers(df_temp)
#     df_temp = replace_guest_popularity(df_temp)
#     df_temp = replace_episode_length(df_temp)
#     df_temp = replace_num_ads(df_temp)

#     df_temp["Episode_Title"] = df_temp.apply(lambda x: x["Episode_Title"] if not str(x["Episode_Title"]).isdigit() else "Episode " + str(x["Episode_Title"]), axis=1)
    
#     return df_temp

In [4]:
from sklearn.preprocessing import StandardScaler

num_features = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage", 
                "Number_of_Ads"] # Removed Outcome var

# numerical features actually transformed
num_features_transform = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage"]

cat_features = ["Podcast_Name", "Genre", "Publication_Day", "Publication_Time", 
                "Episode_Sentiment"] # Episode Title removed since its now int

# This thing not used
features = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage", 
            "Number_of_Ads", "episode_number", "Podcast_Name", "Genre", "Publication_Day", 
            "Publication_Time", "Episode_Sentiment", "Episode_Length_missing"]

def make_cool_features(df):
    df_temp = df.copy()

    # Apply the function to create new column for episode number
    df_temp['episode_number'] = df_temp['Episode_Title'].str.extract(r'Episode (\d+)').astype(int)
    df_temp = df_temp.drop(columns=["Episode_Title"])

    # standardizationfor numerical features
    scaler = StandardScaler()
    df_temp[num_features_transform] = scaler.fit_transform(df_temp[num_features_transform])

    # need this thing
    df_temp1 = df_temp.copy()

    # encoding categorical features
    df_temp = pd.get_dummies(df_temp, columns=cat_features)

    # log transformations
    for feature in num_features_transform:
        if (df_temp[feature] <= 0).any():
            shift_value = abs(df_temp[feature].min()) + 1
            df_temp[f'{feature}_log'] = np.log(df_temp[feature] + shift_value)
        else:
            df_temp[f'{feature}_log'] = np.log(df_temp[feature])

    # ah
    if "Listening_Time_minutes" in df_temp.columns:
        numerical_columns = df_temp.drop(columns=["Listening_Time_minutes"]).select_dtypes(include=['int64', 'float64']).columns
    else:
        numerical_columns = df_temp.select_dtypes(include=['int64', 'float64']).columns
    categorical_columns = df_temp.select_dtypes(include=['object', 'bool']).columns.tolist()

    # interaction temrs
    # interaction terms between numeric and cat
    # for num_feat in numerical_columns:
    #     for cat_feat in categorical_columns:
    #         # interaction term
    #         df_temp[f"{num_feat}_{cat_feat}"] = df_temp[num_feat] * df_temp[cat_feat]

    # interaction terms between all numeric
    for i in range(len(numerical_columns) - 1):
        for j in range(i+1, len(numerical_columns)):
            df_temp[f"{numerical_columns[i]}_{numerical_columns[j]}"] = df_temp[numerical_columns[i]] * df_temp[numerical_columns[j]]

    continuous_num = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage"]
    everything_else = ['Podcast_Name', 'Genre', 'Publication_Day', 'Publication_Time', 'Number_of_Ads', 'Episode_Sentiment', "episode_number"]

    # funky term
    # featureX - Groupby(featureY)[featureX].mean()
    for feat_X in continuous_num:
        for feat_y in everything_else:
            df_temp[f"cool_{feat_X}_{feat_y}"] = df_temp1[feat_X] - df_temp1.groupby(feat_y)[feat_X].transform('mean')
            
    
    return df_temp

In [5]:
%%time
# df_cool = clean_data(df)
df_cool = make_cool_features(df)
all_features = df_cool.drop(columns=["Listening_Time_minutes"]).columns
df_cool.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Columns: 139 entries, id to cool_Guest_Popularity_percentage_episode_number
dtypes: bool(72), float64(64), int64(3)
memory usage: 434.9 MB
CPU times: user 3.25 s, sys: 448 ms, total: 3.7 s
Wall time: 3.65 s


# Model Building

In [6]:
# imports
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer

In [7]:
from sklearn.model_selection import train_test_split
df_baseline = df.copy()
df_baseline[df_baseline.select_dtypes(include='object').columns] = df_baseline[df_baseline.select_dtypes(include='object').columns].astype('category')

X = df_baseline.drop(columns=['Listening_Time_minutes'])
y = df_baseline['Listening_Time_minutes']
X_train, X_val, y_train, y_val = train_test_split(df_baseline.drop(columns=['Listening_Time_minutes']), df_baseline["Listening_Time_minutes"], test_size = 0.3, random_state = 16)

In [8]:
def rmse_loss(y_pred, y_actual):
    return np.sqrt(mean_squared_error(y_pred, y_actual))

rmse = make_scorer(rmse_loss, greater_is_better=False)

In [9]:
%%time
xgb_baseline = xgb.XGBRegressor(enable_categorical=True).fit(X_train, y_train)

y_pred = xgb_baseline.predict(X_val)
cool_score = rmse_loss(y_pred, y_val)
print(f"Baseline RMSE: {cool_score}")

Baseline RMSE: 13.100228571419084
CPU times: user 26.5 s, sys: 69.9 ms, total: 26.6 s
Wall time: 6.84 s


## Feature Selection

In [10]:
# # PCA
# from sklearn.decomposition import PCA

# pca = PCA()
# df_cool_values = df_cool.drop(columns=["Listening_Time_minutes"]).dropna().values
# pca.fit(df_cool_values)
# cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
# n_components = np.argmax(cumulative_explained_variance >= 0.95) + 1
# pca = PCA(n_components=n_components)
# pca_reduced = pca.fit_transform(df_cool_values)

In [11]:
# from sklearn.feature_selection import SequentialFeatureSelector

In [12]:
# %%time
X = df_cool.drop(columns=["Listening_Time_minutes"])
y = df_cool["Listening_Time_minutes"]
# xgb_unfitted = xgb.XGBRegressor()
# sfs = SequentialFeatureSelector(xgb_unfitted, scoring=rmse, n_features_to_select='auto', tol=0.5)
# sfs.fit(X, y)
# selected_features = sfs.get_feature_names_out()
# print(f"Selected Features: {selected_features}")

In [13]:
# df_cool_selected = df_cool[selected_features]

# X_train, X_val, y_train, y_val = train_test_split(df_cool[selected_features], df_cool["Listening_Time_minutes"], test_size=0.3, random_state=16)

# xgb_1 = xgb.XGBRegressor().fit(X_train, y_train)

# y_pred = xgb_1.predict(X_val)
# cool_score = rmse_loss(y_pred, y_val)
# print(f"Feature Selection Only RMSE: {cool_score}")

## Big Tuna

In [14]:
import optuna
from sklearn.model_selection import cross_val_score

In [15]:
def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "random_state": 4
    }
    
    model = xgb.XGBRegressor(**params)
    score = cross_val_score(model, X, y, cv=2, scoring=rmse).mean()
    return np.abs(score)

In [16]:
study = optuna.create_study(direction='minimize',
                            sampler = optuna.samplers.RandomSampler(seed=42),
                            study_name = "BIG BLUE FIN TUNA!!")

[I 2025-04-26 16:54:37,849] A new study created in memory with name: BIG BLUE FIN TUNA!!


In [17]:
%%time
study.optimize(objective, n_trials=100, show_progress_bar=True)

  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-04-26 16:57:45,656] Trial 0 finished with value: 13.570135732102141 and parameters: {'learning_rate': 0.05, 'max_depth': 20, 'subsample': 0.9, 'colsample_bytree': 0.8, 'max_bin': 535, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'lambda': 2.9154431891537547, 'alpha': 0.2537815508265665, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 13.570135732102141.
[I 2025-04-26 16:59:14,998] Trial 1 finished with value: 13.693833692680606 and parameters: {'learning_rate': 0.1, 'max_depth': 18, 'subsample': 0.7, 'colsample_bytree': 0.6, 'max_bin': 584, 'min_child_weight': 4, 'gamma': 2.6237821581611893, 'lambda': 0.05342937261279776, 'alpha': 0.014618962793704957, 'grow_policy': 'depthwise'}. Best is trial 0 with value: 13.570135732102141.
[I 2025-04-26 16:59:53,061] Trial 2 finished with value: 13.115315805676275 and parameters: {'learning_rate': 0.05, 'max_depth': 10, 'subsample': 0.8, 'colsample_bytree': 0.9, 'max_bin': 614, 'min_child_weight': 6, 'gamma': 2.962072844310

In [18]:
best_params = study.best_params
print(f'Best Trial Params: {best_params}')

print(f'Best Trial Value: {study.best_trial.value}')

Best Trial Params: {'learning_rate': 0.05, 'max_depth': 10, 'subsample': 0.8, 'colsample_bytree': 0.9, 'max_bin': 614, 'min_child_weight': 6, 'gamma': 2.9620728443102124, 'lambda': 0.0015339162591163618, 'alpha': 0.26926469100861794, 'grow_policy': 'depthwise'}
Best Trial Value: 13.115315805676275


In [19]:
optuna.visualization.plot_optimization_history(study)

In [20]:
optuna.visualization.plot_parallel_coordinate(study)

In [21]:
optuna.visualization.plot_slice(study, params = ['max_depth', 'gamma', 'min_child_weight', 'alpha', 'subsample'])

In [22]:
optuna.visualization.plot_param_importances(study)

# Submission

In [23]:
best_model = xgb.XGBRegressor(params=best_params, 
                              objective="reg:squarederror",
                              eval_metric= "rmse",
                              tree_method= "gpu_hist",
                              predictor="gpu_predictor").fit(X, y)

In [24]:
df_test = pd.read_csv("/kaggle/input/playground-series-s5e4/test.csv")
df_test[df_test.select_dtypes(include='object').columns] = df_test[df_test.select_dtypes(include='object').columns].astype('category')

In [25]:
# df_test_clean = clean_test_set(df_test)
df_test_cool = make_cool_features(df_test)

In [26]:
cols_to_drop = [col for col in df_test_cool.columns if '20' in col or '100' in col]
df_test_cool = df_test_cool.drop(columns=cols_to_drop)

In [27]:
submission = pd.read_csv("/kaggle/input/playground-series-s5e4/sample_submission.csv")
submission["Listening_Time_minutes"] = best_model.predict(df_test_cool)

In [28]:
submission.to_csv("submission.csv", index=False)
submission.head()

,id,Listening_Time_minutes
0,750000,50.568573
1,750001,47.268456
2,750002,53.185898
3,750003,50.527401
4,750004,44.391212
